Some imports and paths

In [1]:
import os
import pandas as pd
from icecream import ic
from pathlib import Path
from pandas import DataFrame

THIS_FILE_PATH = Path(os.getcwd()) / "olistbr.ipynb"

THIS_PROJECT_PATH = THIS_FILE_PATH.parent.parent
DATA_RAW_PATH = THIS_PROJECT_PATH / "data" / "raw"

if not DATA_RAW_PATH.exists():
    raise FileExistsError()

ic(THIS_PROJECT_PATH)
ic(THIS_FILE_PATH)
ic(DATA_RAW_PATH)

ic| THIS_PROJECT_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr')
ic| THIS_FILE_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/notebooks/olistbr.ipynb')
ic| DATA_RAW_PATH: WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')


WindowsPath('c:/Users/alain/Dev/personal/data-projects/dp-projects/static/olistbr/data/raw')

In [2]:
raw_filenames: list[str] = os.listdir(DATA_RAW_PATH)
raw_filepaths: list[Path] = [DATA_RAW_PATH / f for f in raw_filenames]
# ic(raw_filenames)
# ic(raw_filepaths)

dfs: dict[str, DataFrame] = {f.name: pd.read_csv(f, dtype=str) for f in raw_filepaths}
# ic(dfs)

Measure Functions

In [3]:
def lev_d(a: str, b: str) -> int:
    """Levenshtein distance
    measures the distance between two strings from doing operations:
        insert
        remove
        replace

    time: O(mn) quadratic
    """
    lena = len(a)
    lenb = len(b)

    if lenb == 0:
        return lena

    if lena == 0:
        return lenb

    heada = a[0]
    headb = b[0]
    taila = a[1:]
    tailb = b[1:]

    if heada == headb:
        return lev_d(taila, tailb)

    return 1 + min(lev_d(taila, b), lev_d(a, tailb), lev_d(taila, tailb))


def sim_norm_lev_d(a: str, b: str) -> float:
    """similarity normalised lev_d"""
    lena = len(a)
    lenb = len(b)

    return 1 - lev_d(a, b) / max(lena, lenb)


def dice_sim(A: set[str], B: set[str]) -> float:
    """Dice similarity
    |AnB| / (|A|+|B|/2) = 2*|AnB| / (|A|+|B|) in [0,1]
    how much a set overlaps another set given their average size
    complete coverage = 1
    partial coverage = (0, 1)
    no coverage = 0
    """
    return 2 * len(A.intersection(B)) / (len(A) + len(B))


def get_max_ps(col: pd.Series) -> tuple[int, int]:
    """get max precision and scale from a series
    keep all entries as strings, count their lenghts, find max
    turning to float lose precision
    """
    abs_col: pd.Series = col.astype(str).str.replace("-", "")
    split: pd.Series = abs_col.str.split(".")

    fractional_part: pd.Series = split.str[1]

    max_scale: int = fractional_part.str.len().max()
    max_precision: int = abs_col.str.replace(".", "").str.len().max()
    return (int(max_precision), int(max_scale))

Inference Function

In [ ]:
import string


def infer_dtypes(df: DataFrame) -> DataFrame:
    """
    df -> out_df
    originally intended to fully infer (end-to-end) a cols datatype from its attributes
    incomplete
    currently being used as an aid for the engineer to determine col datatypes from its attributes
    can compartmentalise
    """

    def is_chars_in_string(chars: str, parent_string: str) -> bool:
        if set(chars).intersection(set(parent_string)):
            return True

        return False

    # TODO: add: chars_used_subset_of_hex_digits
    # TODO: add: is_numeric_float
    out_df = pd.DataFrame(
        index=df.columns,
        columns=[
            # keys
            "has_unique_entries",
            # nulls
            "has_nulls",
            "where_nulls",
            "total_nulls",
            # chars
            "sorted_chars_used",
            "total_unique_chars_used",
            "has_ascii",
            "has_non_ascii",
            "has_prefix_zero",
            "dice_sim_to_ascii",
            "dice_sim_to_non_ascii",
            "min_str_value",
            "max_str_value",
            "entry_lengths",
            "total_unique_entry_lengths",
            "max_entry_length",
            "is_fixed_length",
            # numeric
            "chars_used_subset_of_numeric",
            "has_prefix_dash",
            "has_digits",
            "has_hex_digits",
            "has_decimal",
            "dice_sim_to_digits",
            "dice_sim_to_hex_digits",
            "min_numeric_value",
            "max_numeric_value",
            # datetime
            "has_dash",
            "has_colon",
            "has_space",
            # bit
            "has_exactly_two_entries",
        ],
    )

    cols = df.columns

    for col in cols:
        print(col)

        # global vars
        clean_series = df[col].dropna()
        chars_used: set[str] = set("".join(clean_series.astype(str)))
        sorted_chars_used: str = "".join(sorted(chars_used))

        # keys ==================================================
        out_df.loc[col, "has_unique_entries"] = 1 if clean_series.is_unique else 0

        # nulls ==================================================
        where_null = df[col].isnull()

        out_df.loc[col, "has_nulls"] = 1 if where_null.any() else 0
        out_df.loc[col, "where_nulls"] = df[where_null].index.tolist()
        out_df.loc[col, "total_nulls"] = where_null.sum()

        # chars ==================================================
        # if col chars > ascii when col chars - ascii > 0
        excess_ascii: set[str] = chars_used - set(string.printable)
        entry_lengths = sorted(clean_series.astype(str).str.len().unique().tolist())

        out_df.loc[col, "sorted_chars_used"] = sorted_chars_used
        out_df.loc[col, "total_unique_chars_used"] = len(sorted_chars_used)
        out_df.loc[col, "has_ascii"] = (
            1
            if is_chars_in_string(
                string.printable,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_non_ascii"] = 1 if excess_ascii else 0
        out_df.loc[col, "has_prefix_zero"] = (
            1 if clean_series.astype(str).str.startswith("0").any() else 0
        )
        out_df.loc[col, "dice_sim_to_ascii"] = dice_sim(
            set(string.printable),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_non_ascii"] = dice_sim(
            excess_ascii,
            chars_used,
        )
        out_df.loc[col, "min_str_value"] = min(clean_series)
        out_df.loc[col, "max_str_value"] = max(clean_series)
        out_df.loc[col, "entry_lengths"] = entry_lengths
        out_df.loc[col, "total_unique_entry_lengths"] = len(entry_lengths)
        out_df.loc[col, "max_entry_length"] = max(entry_lengths)
        out_df.loc[col, "is_fixed_length"] = 1 if len(entry_lengths) == 1 else 0

        # numeric ==================================================
        str_numeric = string.digits + "-."

        out_df.loc[col, "chars_used_subset_of_numeric"] = (
            1 if chars_used.issubset(str_numeric) else 0
        )
        out_df.loc[col, "has_prefix_dash"] = (
            1 if clean_series.astype(str).str.startswith("-").any() else 0
        )
        out_df.loc[col, "has_digits"] = (
            1
            if is_chars_in_string(
                string.digits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_hex_digits"] = (
            1
            if is_chars_in_string(
                string.hexdigits,
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_decimal"] = (
            1
            if is_chars_in_string(
                ".",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "dice_sim_to_digits"] = dice_sim(
            set(string.digits),
            chars_used,
        )
        out_df.loc[col, "dice_sim_to_hex_digits"] = dice_sim(
            set(string.hexdigits),
            chars_used,
        )

        val = pd.to_numeric(clean_series, errors="coerce")
        numeric_min = val.min()
        numeric_max = val.max()
        out_df.loc[col, "min_numeric_value"] = (
            numeric_min if pd.notnull(numeric_min) else pd.NA
        )
        out_df.loc[col, "max_numeric_value"] = (
            numeric_max if pd.notnull(numeric_max) else pd.NA
        )

        # datetime ==================================================
        out_df.loc[col, "has_dash"] = (
            1
            if is_chars_in_string(
                "-",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_colon"] = (
            1
            if is_chars_in_string(
                ":",
                sorted_chars_used,
            )
            else 0
        )
        out_df.loc[col, "has_space"] = (
            1
            if is_chars_in_string(
                " ",
                sorted_chars_used,
            )
            else 0
        )

        # bit ==================================================
        unique_entries = clean_series.unique()
        out_df.loc[col, "has_exactly_two_entries"] = (
            1 if len(unique_entries) == 2 else 0
        )
    return out_df

Column analysis

In [5]:
pd.set_option("display.max_columns", None)

# filename = raw_filenames[6]
filename = "olist_" + "closed_deals" + "_dataset.csv"
# filename = 'product_category_name_translation.csv'
print(filename)

df = dfs[filename]

df_attributes = infer_dtypes(df)
df_attributes
# print(df_attributes)

olist_closed_deals_dataset.csv
mql_id
seller_id
sdr_id
sr_id
won_date
business_segment
lead_type
lead_behaviour_profile
has_company
has_gtin
average_stock
business_type
declared_product_catalog_size
declared_monthly_revenue


,has_unique_entries,has_nulls,where_nulls,total_nulls,sorted_chars_used,total_unique_chars_used,has_ascii,has_non_ascii,has_prefix_zero,dice_sim_to_ascii,dice_sim_to_non_ascii,min_str_value,max_str_value,entry_lengths,total_unique_entry_lengths,max_entry_length,is_fixed_length,chars_used_subset_of_numeric,has_prefix_dash,has_digits,has_hex_digits,has_decimal,dice_sim_to_digits,dice_sim_to_hex_digits,min_numeric_value,max_numeric_value,has_dash,has_colon,has_space,has_exactly_two_entries
mql_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,000dd3543ac84d906eae52e7c779bb2a,fff8db9478d2fd72df65a67ee6b62f67,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
seller_id,1,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,00065220becb8785e2cf78355eb9bf68,ffc470761de7d0232558ba5e786e57b7,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
sdr_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,068066e24f0c643eb1d089c7dd20cd73,fdb16d3cbbeb5798f2f66c4096be026d,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
sr_id,0,0,[],0,0123456789abcdef,16,1,0,1,0.275862,0.0,060c0a26f19f4d66b42e0d8796688490,fbf4aef3f6915dc0c3c97d6812522f6a,[32],1,32,1,0,0,1,1,0,0.769231,0.842105,<NA>,<NA>,0,0,0,0
won_date,0,0,[],0,-0123456789:,13,1,0,0,0.230088,0.0,2017-12-05 02:00:00,2018-11-14 18:04:19,[19],1,19,1,0,0,1,1,0,0.869565,0.571429,<NA>,<NA>,1,1,1,0
business_segment,0,1,[186],1,_abcdefghijklmnoprstuvwy,24,1,0,0,0.387097,0.0,air_conditioning,watches,"[3, 4, 5, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17...",17,31,0,0,0,0,1,0,0.0,0.26087,<NA>,<NA>,0,0,0,0
lead_type,0,1,"[96, 393, 445, 447, 594, 829]",6,_abdefghilmnoprstuy,19,1,0,0,0.319328,0.0,industry,other,"[5, 7, 8, 10, 12, 13, 15]",7,15,0,0,0,0,1,0,0.0,0.243902,<NA>,<NA>,0,0,0,0
lead_behaviour_profile,0,1,"[3, 5, 17, 20, 22, 23, 24, 25, 26, 38, 39, 47,...",177,",acefghklorstw",15,1,0,0,0.26087,0.0,cat,wolf,"[3, 4, 5, 9, 10, 11]",6,11,0,0,0,0,1,0,0.0,0.216216,<NA>,<NA>,0,0,1,0
has_company,0,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",779,FTaelrsu,8,1,0,0,0.148148,0.0,False,True,"[4, 5]",2,5,0,0,0,0,1,0,0.0,0.2,<NA>,<NA>,0,0,0,1
has_gtin,0,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",778,FTaelrsu,8,1,0,0,0.148148,0.0,False,True,"[4, 5]",2,5,0,0,0,0,1,0,0.0,0.2,<NA>,<NA>,0,0,0,1


Get precision and scale

In [6]:
col_name = "declared_product_catalog_size"
col: pd.Series = df[col_name]

(p, s) = get_max_ps(col)
print(f"{col_name} DECIMAL({p}, {s})")

declared_product_catalog_size DECIMAL(5, 1)


Check if composite cols are unique

In [7]:
def are_cols_unique(*args: DataFrame) -> bool:
    """are columns unique"""
    pass


df = dfs["olist_" + "geolocation" + "_dataset.csv"]
# are these attributes unique?
# if there are any duplicates then it is not unique
cols_list = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
]
# cols_list = ['customer_zip_code_prefix']
# cols_is_unique = False if df[cols_list].duplicated().any() else True
# print(cols_is_unique)

a = df["geolocation_zip_code_prefix"].unique()
print(sorted(a))

['01001', '01002', '01003', '01004', '01005', '01006', '01007', '01008', '01009', '01010', '01011', '01012', '01013', '01014', '01015', '01016', '01017', '01018', '01019', '01020', '01021', '01022', '01023', '01024', '01025', '01026', '01027', '01028', '01029', '01030', '01031', '01032', '01033', '01034', '01035', '01036', '01037', '01038', '01039', '01040', '01041', '01042', '01043', '01044', '01045', '01046', '01047', '01048', '01049', '01050', '01101', '01102', '01103', '01104', '01105', '01106', '01107', '01108', '01109', '01120', '01121', '01122', '01123', '01124', '01125', '01126', '01127', '01128', '01129', '01130', '01131', '01132', '01133', '01134', '01135', '01136', '01137', '01138', '01139', '01140', '01141', '01142', '01144', '01150', '01151', '01152', '01153', '01154', '01155', '01156', '01189', '01200', '01201', '01202', '01203', '01204', '01205', '01206', '01207', '01208', '01209', '01210', '01211', '01212', '01213', '01214', '01215', '01216', '01217', '01218', '01219', 

Notes

In [8]:
"""scratch
inclusive
tinyint:     2**0-1 to 2**8-1
smallint:   -2**15  to 2**15-1
int:        -2**31  to 2**31-1
bigint:     -2**63  to 2**63-1

money: -922,337,203,685,477.5808 to 922,337,203,685,477.5807 (-922,337,203,685,477.58
to 922,337,203,685,477.58 for Informatica. Informatica only supports two decimals, not four.)

smallmoney: -214,748.3648 to 214,748.3647

prefer decimal over money
"""

"""
different known datatypes

numeric if chars_used_subset_of_numeric else other
    exact numerics
        (tiny, small, int, big)
        money
        smallmoney
    approximate numerics
        float
        real

datetime if has_dash and has_colon and has_space else other
    TODO: has_datetime_format
    datetime2

bit if has_exactly_two_entries else other
    TODO: has_bit_format: True/False, 1/0, true/false, yes/no
    bit

text is other 

text
numeric
datetime
bit

assume is text
(n, var, char) (#)
    'n' if has_non_ascii else ''
    '' if is_fixed_length else 'var'
    'char'
    # = max_entry_length

assume is int
(tiny, small, int, big)
    has_prefix_dash
    M = max(abs(min_numeric_value), abs(max_numeric_value))
    
    if min_numeric_value is negative: cannot be tinyint

    if M <= 2**8-1:

decimal(p,s)
datetime2
bit

not null
primary key

"""

"\ndifferent known datatypes\n\nnumeric if chars_used_subset_of_numeric else other\n    exact numerics\n        (tiny, small, int, big)\n        money\n        smallmoney\n    approximate numerics\n        float\n        real\n\ndatetime if has_dash and has_colon and has_space else other\n    TODO: has_datetime_format\n    datetime2\n\nbit if has_exactly_two_entries else other\n    TODO: has_bit_format: True/False, 1/0, true/false, yes/no\n    bit\n\ntext is other \n\ntext\nnumeric\ndatetime\nbit\n\nassume is text\n(n, var, char) (#)\n    'n' if has_non_ascii else ''\n    '' if is_fixed_length else 'var'\n    'char'\n    # = max_entry_length\n\nassume is int\n(tiny, small, int, big)\n    has_prefix_dash\n    M = max(abs(min_numeric_value), abs(max_numeric_value))\n\n    if min_numeric_value is negative: cannot be tinyint\n\n    if M <= 2**8-1:\n\ndecimal(p,s)\ndatetime2\nbit\n\nnot null\nprimary key\n\n"